In [ ]:
import os

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader
from word2vec import Word2Vec
from utils import (
    SkipGramPairsDataset,
    cosseno,
    score_para_grau,
    token_para_id
)

In [ ]:
BATCH_SIZE = 256
EMBEDDING_DIM = 300
NEGATIVE_SAMPLES = 5
EPOCHS = 5
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
base_dir = Path.cwd()
pares_path = os.path.join(base_dir, "..","datasets", "pares_skipgram_direito.csv")
print("Carregando os pares de palavras...")
print("Pares path:", pares_path)

df_pares = pd.read_csv(pares_path)
df_pares = df_pares[(df_pares["centro"] != 0) & (df_pares["contexto"] != 0)].reset_index(drop=True)

VOCAB_SIZE = int(max(df_pares["centro"].max(), df_pares["contexto"].max())) + 1

print("Device:", DEVICE)
print("VOCAB_SIZE:", VOCAB_SIZE)
print("Quantidade de pares:", len(df_pares))

In [ ]:
df_corpus_direito = pd.read_csv(os.path.join(base_dir, "..","datasets", "corpus_direito_tratado.csv"))
df_corpus_direito.head(5)

In [ ]:
dataset = SkipGramPairsDataset(df_pares)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

if len(dataset) == 0:
    raise ValueError("Dataset de pares está vazio após o filtro de UNK.")

In [ ]:
def sample_negative_ids(batch_size, num_negatives, vocab_size, device, positive_ids=None):
    negatives = torch.randint(
        low=1,
        high=vocab_size,
        size=(batch_size, num_negatives),
        device=device,
    )

    if positive_ids is not None:
        positive_ids = positive_ids.unsqueeze(1)
        mask = negatives.eq(positive_ids)
        while mask.any():
            negatives[mask] = torch.randint(
                low=1,
                high=vocab_size,
                size=(mask.sum().item(),),
                device=device,
            )
            mask = negatives.eq(positive_ids)

    return negatives

In [ ]:
word2vec_model = Word2Vec(vocab_size=VOCAB_SIZE, embedding_dim=EMBEDDING_DIM).to(DEVICE)
optimizer = torch.optim.Adam(word2vec_model.parameters(), lr=LEARNING_RATE)

In [ ]:
loss_history = []

word2vec_model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0

    for center_ids, pos_context_ids in loader:
        center_ids = center_ids.to(DEVICE)
        pos_context_ids = pos_context_ids.to(DEVICE)

        neg_context_ids = sample_negative_ids(
            batch_size=center_ids.size(0),
            num_negatives=NEGATIVE_SAMPLES,
            vocab_size=VOCAB_SIZE,
            device=DEVICE,
            positive_ids=pos_context_ids,
        )

        optimizer.zero_grad()
        loss = word2vec_model(center_ids, pos_context_ids, neg_context_ids)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    epoch_loss = total_loss / len(loader)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} - loss média: {epoch_loss:.4f}")

In [ ]:
output_dir = base_dir.parent / "datasets"
output_dir.mkdir(parents=True, exist_ok=True)

torch.save(word2vec_model.state_dict(), output_dir / "word2vec_model.pt")
torch.save(word2vec_model.get_input_embeddings().detach().cpu(), output_dir / "word_embeddings_input.pt")
torch.save(word2vec_model.get_output_embeddings().detach().cpu(), output_dir / "word_embeddings_output.pt")

pd.DataFrame({"epoch_loss": loss_history}).to_csv(output_dir / "word2vec_loss_history.csv", index=False)

print("Modelo, embeddings e histórico de loss salvos.")

### Uso do Word2Vec para o cálculo de similaridade

#### Direito

In [ ]:
# Loading do dataset de similaridade de direito

direito_path = os.path.join(base_dir, "..","datasets", "similarity_direito_processed.csv")
print("Carregando o dataset de similaridade de direito...")
print("Dataset path:", direito_path)

if os.path.exists(direito_path):
    df_similarity = pd.read_csv(direito_path)
    print("Dataset de similaridade carregado com sucesso.")
else:
    print("Dataset de similaridade não encontrado. Verifique o caminho e tente novamente.")

In [ ]:
# Carregar vocabulário salvo no treino
with open(output_dir / "vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)


df_similarity["id1"] = df_similarity["token1_norm"].apply(lambda token: token_para_id(vocab, token))
df_similarity["id2"] = df_similarity["token2_norm"].apply(lambda token: token_para_id(vocab, token))

df_similarity["tem_unk"] = (df_similarity["id1"] == 0) | (df_similarity["id2"] == 0)

emb_input = word2vec_model.get_input_embeddings().detach().cpu()

similarity_scores = []
for _, row in df_similarity.iterrows():
    if row["tem_unk"]:
        similarity_scores.append(np.nan)
        continue

    v1 = emb_input[row["id1"]].numpy()
    v2 = emb_input[row["id2"]].numpy()

    cos = cosseno(v1, v2)
    score_01 = (cos + 1.0) / 2.0
    similarity_scores.append(score_01)

df_similarity["similarity_score"] = similarity_scores
df_similarity["similarity_grau"] = df_similarity["similarity_score"].apply(
    lambda x: np.nan if pd.isna(x) else score_para_grau(x)
)

df_similarity[[
    "token1", "token2",
    "token1_norm", "token2_norm",
    "id1", "id2",
    "similarity_score",
    "similarity_grau"
]].head()